# End-to-End Multi-Agent Project
## AI-Powered Self-Healing Data Pipeline on Databricks
### Business Challenge
Traditional data pipelines are typically built using predefined rules and hardcoded transformations. While effective for fixed use cases, they often struggle when data structures change or new datasets are introduced.
Common challenges include:
- High manual effort for development and maintenance
- Frequent failures caused by schema evolution and data quality issues
- Time-consuming debugging and operational support
- Dataset-specific logic that limits scalability and reusability
As data ecosystems continue to grow, maintaining these pipelines becomes increasingly complex and resource-intensive.
---
## Objective
Build an AI-driven, self-healing data pipeline that can:
- Automatically ingest and process data
- Perform intelligent data cleaning and transformation
- Adapt to different datasets without manual code changes
- Detect failures, generate fixes, and retry execution autonomously
> The objective is to evolve from traditional rule-based data engineering toward intelligent, agent-driven data operations.
---
## Overview
This project demonstrates a Self-Healing Agentic Data Pipeline built on Databricks using Large Language Models (LLMs) and AI agents.
Instead of relying on static ETL workflows, the solution leverages intelligent agents that can analyze data, generate processing logic, execute transformations, and recover from failures automatically.
The platform enables:
- Dynamic PySpark code generation
- Automated ingestion and transformation workflows
- Intelligent error detection and monitoring
- Self-healing recovery through automated remediation
---
## Core Concept
> **Enable AI agents to operate like Data Engineers — generating, executing, validating, and repairing data pipelines automatically.**
---
## Key Features
- Dynamic data ingestion without hardcoded rules
- Dataset-agnostic data cleansing and transformation
- LLM-generated PySpark processing logic
- Automated error diagnosis and self-healing retries
- Scalable multi-agent architecture
---
## Why This Project?
### Traditional Data Pipelines
- Require extensive manual coding
- Need continuous maintenance and updates
- Are vulnerable to schema and data changes
- Become difficult to scale across multiple datasets
### Agentic AI-Powered Pipelines
- Adapt automatically to evolving data
- Reduce engineering effort and maintenance overhead
- Recover intelligently from failures
- Improve scalability and operational efficiency
By combining Databricks, AI agents, and LLM-driven code generation, this project demonstrates a modern approach to building resilient, adaptive, and self-managing data pipelines.
 


## A single flow diagram captures the entire process:

                    ┌─────────────────────┐
                    │   Source Dataset    │
                    │    CSV / Parquet    │
                    └──────────┬──────────┘
                               │
                               ▼
                 ┌──────────────────────────┐
                 │   Ingestion Agent        │
                 │ Analyze Schema & Metadata│
                 └──────────┬───────────────┘
                            │
                            ▼
                 ┌──────────────────────────┐
                 │ Transformation Agent     │
                 │ Generate PySpark Code    │
                 │ using LLM                │
                 └──────────┬───────────────┘
                            │
                            ▼
                 ┌──────────────────────────┐
                 │ Execute on Databricks    │
                 │ Spark Runtime            │
                 └──────────┬───────────────┘
                            │
                  ┌─────────▼─────────┐
                  │ Execution Success?│
                  └───────┬─────┬─────┘
                          │Yes  │No
                          │     │
                          ▼     ▼
             ┌─────────────────┐   ┌──────────────────────┐
             │ Curated Output  │   │ Self-Healing Agent   │
             │ Delta Table     │   │ Analyze Error Logs   │
             └─────────────────┘   │ Fix Generated Code   │
                                   │ Retry Execution      │
                                   └──────────┬───────────┘
                                              │
                                              ▼
                                   ┌──────────────────────┐
                                   │ Re-Execute Pipeline  │
                                   └──────────────────────┘
 

In [0]:
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key="",
    api_version="2024-12-01-preview",
    azure_endpoint="",
)


def invoke_llm(prompt_text):
    llm_response = client.chat.completions.create(
        model="gpt-4o", messages=[{"role": "user", "content": prompt_text}]
    )

    generated_code = llm_response.choices[0].message.content.strip()

    # sanitize model output

    if generated_code.lower().startswith("python"):
        generated_code = generated_code[len("python") :].strip()

    generated_code = generated_code.replace("```python", "").replace("```", "").strip()

    return generated_code


# ================================

# Bronze Loader Orchestrator

# ================================


def run_bronze_ingestion(volume_root, target_catalog, target_schema, max_retries=3):
    print("\n[START] Bronze ingestion pipeline initiated")

    print("[INFO] Source volume:", volume_root)

    base_instruction = f"""

        You are working as a Senior Data Engineer responsible for building a production-grade Bronze ingestion framework in Databricks.
        
        TASK:

        Scan the provided Databricks Volume path and load all supported datasets into Bronze Delta tables.
        
        SOURCE PATH:

        {volume_root}
        
        TARGET STRUCTURE:

        {target_catalog}.{target_schema}.<table_name>
        
        REQUIREMENTS:
        
        1. Discover all files and folders using dbutils.fs.ls().
        
        2. Supported inputs:

        - CSV files

        - Parquet files

        - Parquet dataset directories
        
        3. CSV Handling:

        - spark.read.format("csv")

        - header=True

        - inferSchema=True
        
        4. Parquet Handling:

        - spark.read.parquet()
        
        5. Parquet Directory Handling:

        - If a directory contains parquet data, treat it as a dataset

        - Load using spark.read.parquet(directory_path)

        - Use directory name as table name
        
        6. Table Naming Convention:

        - remove extensions (.csv, .parquet)

        - lowercase

        - replace spaces with "_"

        - replace special characters with "_"

        - do not use csv or parquet in table name
        
        7. Write output as Delta:

        - format("delta")

        - mode("overwrite")

        - option("overwriteSchema","true")
        
        8. Enable schema evolution for parquet:

        - option("mergeSchema","true")
        
        9. Logging requirements:

        Use meaningful operational logs like:

        - "[SCAN] Exploring source volume"

        - "[FOUND] Dataset detected:"

        - "[LOAD] Reading source:"

        - "[WRITE] Writing to Delta table:"

        - "[SUCCESS] Load completed:"

        - "[CLEANUP] Removing processed source:"

        - "[ERROR] Processing failed for:"
        
        10. Cleanup:

        - After successful processing, delete source using dbutils.fs.rm(path, True)
        
        11. Error handling:

        - Each file/directory must be wrapped in try/except

        - Continue processing remaining items even if one fails
        
        12. Empty source validation:

        If no valid CSV or Parquet sources exist:

        - Print: "[EXIT] No supported datasets found in source path"

        - Stop execution immediately using SystemExit
        
        STRICT RULES:

        - Return ONLY Python code

        - No markdown

        - No explanations

        - No comments

        - No f-strings

        - Use string concatenation only

        - Assume spark and dbutils already exist

        - Do not create SparkSession

"""

    attempt = 0

    last_error = None

    generated_code = None

    while attempt < max_retries:
        print("\n[INFO] Generation attempt:", attempt + 1)

        if attempt == 0:
            prompt = base_instruction

        else:
            prompt = f"""

Previous execution failed.
 
FAILED CODE:

{generated_code}
 
ERROR:

{last_error}
 
TASK:

Fix the code while preserving original logic.
 
RULES:

- Return ONLY working Python code

- No explanations

- No markdown

- No f-strings

- Ensure correctness and completeness

"""

        generated_code = invoke_llm(prompt)

        generated_code = (
            generated_code.replace("```python", "").replace("```", "").strip()
        )

        print("\n[GENERATED CODE]\n", generated_code)

        try:
            print("\n[RUNNING] Executing generated pipeline...\n")

            exec(generated_code, globals())

            print("\n[SUCCESS] Bronze ingestion completed successfully")

            return {
                "status": "success",
                "attempts": attempt + 1,
                "code": generated_code,
            }

        except Exception as err:
            last_error = str(err)

            print("\n[FAILURE] Execution error:", last_error)

            attempt += 1

    print("\n[FAILED] All retry attempts exhausted")

    return {
        "status": "failed",
        "error": last_error,
        "final_code": generated_code,
        "attempts": max_retries,
    }

In [0]:
result = run_bronze_ingestion(
    volume_root="/Volumes/agentic_ai_data_ingestion/bronze_ai/bronze_volumne",
    target_catalog="agentic_ai_data_ingestion",
    target_schema="bronze_ai"
)

print(result)

In [0]:
# ================================
#  SILVER LAYER AGENT
# ================================

GENERAL_CLEANING_GUIDELINES = """
You are cleaning data from Bronze to Silver.

IMPORTANT:
- Do NOT blindly apply all transformations
- First understand the data
- Then decide what cleaning steps are actually required

Possible cleaning actions (apply ONLY if needed):

- Standardize column names (lowercase, replace spaces and special characters with underscores)
- Trim leading and trailing whitespace from all string columns
- Replace empty strings and common null representations (NULL, N/A, NA, Unknown) with null values
- Cast columns to appropriate data types based on inferred schema
- Standardize date and timestamp formats
- Remove duplicate records based on business keys or full-row duplicates
- Normalize categorical values for consistent casing and representation
- Validate mandatory fields and remove or quarantine invalid records
- Add audit columns (ingestion_timestamp, processing_timestamp, source_file_name)
- Generate data quality flags for records failing validation checks

RULE:
- Apply ONLY relevant transformations
- Avoid unnecessary changes

APPROACH:
1. Analyze schema and sample data
2. Create a cleaning plan
3. Generate PySpark code based on that plan
"""
def silver_cleaning_agent(catalog, bronze_schema, silver_schema, user_instructions, max_retries=5):

    print("\n Running Smart Silver Cleaning Agent...")

    tables = spark.sql(f"SHOW TABLES IN {catalog}.{bronze_schema}") \
                 .select("tableName").collect()

    table_list = [row.tableName for row in tables]

    print("\n Tables found:", table_list)

    results = []

    for table in table_list:
        print(f"\n Processing table: {table}")

        # Extract schema + sample
        df = spark.table(f"{catalog}.{bronze_schema}.{table}")
        schema_info = df.dtypes
        sample_data = df.limit(5).toPandas().to_dict(orient="records")

        attempt = 0
        last_error = None
        code = None

        while attempt < max_retries:

            print(f"\n Attempt {attempt + 1}...\n")

            if attempt == 0:
                prompt = f"""
                You are a senior data engineer working in Accenture , India

                TABLE: {table}

                SCHEMA:
                {schema_info}

                SAMPLE DATA:
                {sample_data}

                USER GUIDELINES:
                {user_instructions}

                TASK:
                Step 1: Analyze the table
                Step 2: Decide what cleaning is needed (create a plan internally)
                Step 3: Generate PySpark code to implement ONLY required cleaning

                INPUT:
                {catalog}.{bronze_schema}.{table}

                OUTPUT:
                {catalog}.{silver_schema}.{table}

                IMPORTANT:
                - Do NOT apply all cleaning steps blindly
                - Apply only necessary transformations
                - Keep transformations minimal and safe

                STRICT RULES:
                - ONLY Python code
                - NO markdown
                - NO explanations
                - DO NOT use f-strings
                - Use string concatenation
                - DO NOT create SparkSession
                - Code must be COMPLETE and EXECUTABLE
                """

            else:
                prompt = f"""
                The following PySpark code failed.

                FAILED CODE:
                {code}

                ERROR:
                {last_error}

                Fix the code.

                IMPORTANT:
                - Update the logic as needed
                - Ensure syntax is correct
                - Ensure variables are correct
                - Remove invalid text like 'python'

                STRICT RULES:
                - ONLY Python code
                - NO markdown
                - NO explanations
                - Ensure executable code
                """

            code = call_llm(prompt)

            # Clean output
            code = code.replace("```python", "").replace("```", "").strip()
            if code.lower().startswith("python"):
                code = code[len("python"):].strip()

            print("===== GENERATED CODE =====\n", code)

            try:
                print("\n Executing...\n")
                exec(code, globals())

                print(f"\n Success: {table}")

                results.append({
                    "table": table,
                    "status": "success",
                    "attempts": attempt + 1
                })

                break

            except Exception as e:
                last_error = str(e)

                print(f"\n Failed Attempt {attempt + 1}")
                print("Error:", last_error)

                attempt += 1

        if attempt == max_retries:
            results.append({
                "table": table,
                "status": "failed",
                "error": last_error
            })

    print("\n All tables processed!")

    return results

In [0]:
result = silver_cleaning_agent(
    catalog="agentic_ai_data_ingestion",
    bronze_schema="bronze_ai",
    silver_schema="silver_ai",
    user_instructions=GENERAL_CLEANING_GUIDELINES
)

print(result)

In [0]:
%sql
SELECT * FROM agentic_ai_data_ingestion.bronze_ai.bigmart_sales

In [0]:
%sql
SELECT * FROM agentic_ai_data_ingestion.silver_ai.bigmart_sales

In [0]:
# ==================================
# INTERACTIVE DATA PIPELINE 
# ==================================

def launch_data_pipeline_demo():

    print("\n🚀 Intelligent Data Pipeline Assistant")
    print("Would you like to begin the data processing workflow? (yes/no)")

    user_response = input("Enter choice: ").strip().lower()

    if user_response not in ["yes", "y"]:
        print("\nWorkflow cancelled by user.")
        return

    # BRONZE INGESTION AGENT

    print("\n" + "=" * 50)
    print(" Initializing: BRONZE INGESTION AGENT")
    print("=" * 50)

    bronze_result = run_bronze_ingestion(
    volume_root="/Volumes/agentic_ai_data_ingestion/bronze_ai/bronze_volumne",
    target_catalog="agentic_ai_data_ingestion",
    target_schema="bronze_ai"
    )


    print("\nBronze Layer Output:", bronze_result)

    if bronze_result["status"] != "success":
        print("\nBronze stage encountered an issue. Pipeline execution halted.")
        return

    # CONFIRM SILVER STAGE

    print("\nBronze layer processing finished successfully.")
    print("Do you want to continue with the Silver Transformation Agent? (yes/no)")

    user_response = input("Enter choice: ").strip().lower()

    if user_response not in ["yes", "y"]:
        print("\nExecution stopped after Bronze processing.")
        return


    # SILVER TRANSFORMATION AGENT

    print("\n" + "=" * 50)
    print(" Initializing: SILVER TRANSFORMATION AGENT")
    print("=" * 50)

    silver_result = silver_cleaning_agent(
        catalog="agentic_ai_data_ingestion",
        bronze_schema="bronze_ai",
        silver_schema="silver_ai",
        user_instructions=GENERAL_CLEANING_GUIDELINES
    )

    print("\nSilver Layer Output:", silver_result)

    print("\n Data pipeline execution completed successfully!")



# EXECUTE
launch_data_pipeline_demo()

#### DELETING TABLES FROM BRONZE LAYER

In [0]:
tables = spark.sql("SHOW TABLES IN agentic_ai_data_ingestion.bronze_ai").select("tableName").collect()
for row in tables:
    table_name = row.tableName
    spark.sql("DROP TABLE IF EXISTS agentic_ai_data_ingestion.bronze_ai." + table_name)

#### DELETING TABLES FROM SILVER LAYER

In [0]:
tables = spark.sql("SHOW TABLES IN agentic_ai_data_ingestion.silver_ai").select("tableName").collect()
for row in tables:
    table_name = row.tableName
    spark.sql("DROP TABLE IF EXISTS agentic_ai_data_ingestion.silver_ai." + table_name)